# Bayesian NPS Analysis with Cultural Bias Adjustment

This notebook implements a comprehensive Bayesian hierarchical model to adjust NPS scores for cultural biases using Hofstede indices.

## Overview
- Separate Bayesian models for each sentiment signal (Stars, BERT, Gemma-27B, Qwen-32B, Qwen-14B)
- Cultural bias adjustment using Hofstede dimensions
- Country-level random effects
- Comprehensive evaluation metrics and validation
- Composite Reliability Score (CRS-B) calculation


## Configuration


In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Configuration Parameters
CONFIG = {
    # Data paths - UPDATE THESE PATHS TO MATCH YOUR DATA FILE LOCATION
    'data_file': 'nps_dataset.csv',  # Main dataset file with 30,000 reviews
    # Alternative paths to try: '../nps_dataset.csv', 'CodeBase/nps_dataset.csv', etc.
    'hofstede_file': 'hofstede_country_scores.csv',
    
    # Model parameters
    'num_samples': 2000,  # MCMC samples
    'num_warmup': 1000,   # Warmup samples
    'num_chains': 4,      # Number of chains
    'rng_key': 42,        # Random seed
    
    # Prior configurations
    'prior_type': 'regularized',  # 'weakly_informative' or 'regularized'
    
    # Evaluation parameters
    'min_country_reviews': 10,  # Minimum reviews per country for analysis
    
    # CRS-B weights (will be learned, but can set initial values)
    'crs_initial_weights': [0.25, 0.25, 0.25, 0.25],
    
    # Output settings
    'save_plots': True,
    'plot_format': 'png',
    'plot_dpi': 300,
    
    # GPU settings
    'use_gpu': True,
    'device': 'cuda' if os.environ.get('CUDA_VISIBLE_DEVICES') else 'cpu'
}

# Sentiment signal names
SENTIMENT_SIGNALS = {
    'stars': {
        'score_col': 'stars',
        'category_col': 'nps_category_stars',
        'name': 'Raw Stars'
    },
    'bert': {
        'score_col': 'sentiment_score_bert_base',
        'category_col': 'nps_category_bert_base',
        'name': 'BERT Base'
    },
    'gemma27b': {
        'score_col': 'sentiment_score_gemma27b',
        'category_col': 'nps_category_gemma27b',
        'name': 'Gemma-27B'
    },
    'qwen32b': {
        'score_col': 'sentiment_score_qwen25_32b',
        'category_col': 'nps_category_qwen25_32b',
        'name': 'Qwen-32B'
    },
    'qwen14b': {
        'score_col': 'sentiment_score_qwen25_14b',
        'category_col': 'nps_category_qwen25_14b',
        'name': 'Qwen-14B'
    }
}

# Hofstede dimensions
HOFSTEDE_DIMS = ['pdi', 'idv', 'mas', 'uai', 'lto', 'ivr']

print(f"Configuration loaded. Device: {CONFIG['device']}")


## Imports and Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import time
import json
from pathlib import Path
from collections import defaultdict
from scipy import stats
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import cohen_kappa_score
import jax
import jax.numpy as jnp
from jax import random
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, Predictive
from numpyro.contrib.control_flow import scan

# Set JAX backend
if CONFIG['use_gpu']:
    try:
        jax.config.update('jax_platform_name', 'gpu')
        print(f"JAX backend: {jax.devices()}")
    except:
        jax.config.update('jax_platform_name', 'cpu')
        print("GPU not available, using CPU")
else:
    jax.config.update('jax_platform_name', 'cpu')

# Set random seeds
numpyro.set_platform(CONFIG['device'])
numpyro.set_host_device_count(CONFIG['num_chains'])

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

# Create output directory with timestamp
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path(f"bayesian_nps_results_{TIMESTAMP}")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Timestamp: {TIMESTAMP}")

# Start timing
START_TIME = time.time()


## Data Loading and Preprocessing


In [ ]:
def load_and_preprocess_data(data_file, hofstede_file):
    """
    Load and preprocess the NPS dataset.
    
    Returns:
        df: Preprocessed dataframe
        hofstede_df: Hofstede indices dataframe
        country_to_idx: Mapping from country to index
    """
    print(f"Loading data from {data_file}...")
    
    # Try to find the data file
    possible_paths = [
        data_file,
        f"../{data_file}",
        f"../../{data_file}",
        f"CodeBase/{data_file}",
        f"../CodeBase/{data_file}"
    ]
    
    data_path = None
    for path in possible_paths:
        if os.path.exists(path):
            data_path = path
            break
    
    if data_path is None:
        raise FileNotFoundError(f"Data file not found. Tried: {possible_paths}")
    
    df = pd.read_csv(data_path)
    print(f"Loaded {len(df)} reviews")
    
    # Load Hofstede data
    hofstede_paths = [
        hofstede_file,
        f"../{hofstede_file}",
        f"CodeBase/{hofstede_file}",
        f"../CodeBase/{hofstede_file}"
    ]
    
    hofstede_path = None
    for path in hofstede_paths:
        if os.path.exists(path):
            hofstede_path = path
            break
    
    if hofstede_path is None:
        raise FileNotFoundError(f"Hofstede file not found. Tried: {hofstede_paths}")
    
    hofstede_df = pd.read_csv(hofstede_path)
    hofstede_df['country'] = hofstede_df['country'].str.strip().str.title()
    
    # Merge Hofstede data
    if 'country' in df.columns:
        df['country'] = df['country'].str.strip().str.title()
        df = df.merge(hofstede_df, on='country', how='left')
    
    # Filter countries with sufficient reviews
    country_counts = df['country'].value_counts()
    valid_countries = country_counts[country_counts >= CONFIG['min_country_reviews']].index
    df = df[df['country'].isin(valid_countries)].copy()
    
    print(f"After filtering: {len(df)} reviews from {len(valid_countries)} countries")
    
    # Create country mapping
    unique_countries = sorted(df['country'].unique())
    country_to_idx = {country: idx for idx, country in enumerate(unique_countries)}
    df['country_idx'] = df['country'].map(country_to_idx)
    
    # Normalize Hofstede dimensions
    scaler = StandardScaler()
    for dim in HOFSTEDE_DIMS:
        if dim in df.columns:
            # Fill missing values with median
            median_val = df[dim].median()
            df[dim] = df[dim].fillna(median_val)
            # Normalize
            df[f'{dim}_norm'] = scaler.fit_transform(df[[dim]]).flatten()
    
    # Ensure sentiment scores are in valid range [1, 5]
    for signal_name, signal_info in SENTIMENT_SIGNALS.items():
        score_col = signal_info['score_col']
        if score_col in df.columns:
            # Clip to valid range
            df[score_col] = df[score_col].clip(1, 5)
            # Fill missing values with median
            median_score = df[score_col].median()
            df[score_col] = df[score_col].fillna(median_score)
    
    return df, hofstede_df, country_to_idx

# Load data
df, hofstede_df, country_to_idx = load_and_preprocess_data(
    CONFIG['data_file'],
    CONFIG['hofstede_file']
)

print(f"\nData shape: {df.shape}")
print(f"\nColumns: {list(df.columns)[:20]}...")  # Show first 20 columns
print(f"\nCountries: {len(country_to_idx)}")
print(f"\nSample countries: {list(country_to_idx.keys())[:10]}")


## NPS Calculation Functions


In [ ]:
# Import functions
from bayesian_nps_functions import (
    calculate_nps_category, calculate_nps_score, calculate_nps_by_country,
    get_prior_config, bayesian_sentiment_model, fit_bayesian_model,
    calculate_country_nps_dispersion, calculate_instability_reduction,
    calculate_hofstede_correlation, calculate_agreement_score,
    calculate_within_country_consistency, calculate_distribution_alignment,
    calculate_crs_b, prior_sensitivity_analysis, posterior_predictive_check,
    leave_one_country_out_validation, plot_sentiment_distributions,
    plot_nps_comparison, plot_crs_comparison, generate_comprehensive_report
)

# Calculate initial NPS for all signals
initial_nps = {}
for signal_name, signal_info in SENTIMENT_SIGNALS.items():
    score_col = signal_info['score_col']
    category_col = signal_info['category_col']
    
    if score_col in df.columns:
        nps_scores = calculate_nps_by_country(df, score_col, category_col)
        initial_nps[signal_name] = nps_scores
        print(f"{signal_info['name']}: Mean NPS = {nps_scores.mean():.2f}, Std = {nps_scores.std():.2f}")


## Bayesian Model Fitting


In [ ]:
# Fit Bayesian models for each signal
prior_config = get_prior_config(CONFIG['prior_type'])

fitted_models = {}
adjusted_scores_dict = {}
posterior_samples = {}

print("=" * 80)
print("FITTING BAYESIAN MODELS")
print("=" * 80)

for signal_name, signal_info in SENTIMENT_SIGNALS.items():
    print(f"\n{'='*80}")
    print(f"Processing: {signal_info['name']}")
    print(f"{'='*80}")
    
    try:
        mcmc, adjusted_scores, samples = fit_bayesian_model(
            df, signal_name, signal_info, prior_config, CONFIG
        )
        
        if mcmc is not None:
            fitted_models[signal_name] = mcmc
            adjusted_scores_dict[signal_name] = adjusted_scores
            posterior_samples[signal_name] = samples
            
            # Add adjusted scores to dataframe
            df[f'adjusted_{signal_name}'] = adjusted_scores
            
            print(f"✓ {signal_info['name']} model fitted successfully")
            print(f"  Adjusted scores: mean={np.nanmean(adjusted_scores):.3f}, std={np.nanstd(adjusted_scores):.3f}")
        else:
            print(f"✗ {signal_info['name']} model fitting failed")
    except Exception as e:
        print(f"✗ Error fitting {signal_info['name']}: {str(e)}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*80}")
print(f"Fitted {len(fitted_models)} models successfully")
print(f"{'='*80}")


## Calculate Adjusted NPS Scores


In [ ]:
# Calculate NPS from adjusted scores
adjusted_nps = {}

for signal_name, signal_info in SENTIMENT_SIGNALS.items():
    if signal_name in adjusted_scores_dict:
        adjusted_col = f'adjusted_{signal_name}'
        
        # Calculate categories from adjusted scores
        df[f'{adjusted_col}_category'] = df[adjusted_col].apply(calculate_nps_category)
        
        # Calculate NPS by country
        nps_scores = calculate_nps_by_country(df, adjusted_col, f'{adjusted_col}_category')
        adjusted_nps[signal_name] = nps_scores
        
        print(f"{signal_info['name']} (Adjusted):")
        print(f"  Mean NPS = {nps_scores.mean():.2f}")
        print(f"  Std NPS = {nps_scores.std():.2f}")
        print(f"  Range = [{nps_scores.min():.2f}, {nps_scores.max():.2f}]")
        print()


## Evaluation Metrics Calculation


In [ ]:
# Calculate comprehensive evaluation metrics
metrics_results = {}

print("=" * 80)
print("CALCULATING EVALUATION METRICS")
print("=" * 80)

for signal_name, signal_info in SENTIMENT_SIGNALS.items():
    if signal_name not in initial_nps or signal_name not in adjusted_nps:
        continue
    
    print(f"\n{signal_info['name']}:")
    print("-" * 80)
    
    metrics = {}
    
    # 1. Country-wise NPS dispersion (before and after)
    initial_disp = calculate_country_nps_dispersion(initial_nps[signal_name])
    adjusted_disp = calculate_country_nps_dispersion(adjusted_nps[signal_name])
    metrics['country_dispersion_before'] = initial_disp
    metrics['country_dispersion_after'] = adjusted_disp
    metrics['country_dispersion'] = adjusted_disp  # For CRS-B
    
    # 2. Instability reduction
    reduction, std_before, std_after = calculate_instability_reduction(
        initial_nps[signal_name], adjusted_nps[signal_name]
    )
    metrics['instability_reduction'] = reduction
    metrics['std_before'] = std_before
    metrics['std_after'] = std_after
    
    # 3. Correlation with Hofstede dimensions
    hofstede_corrs = {}
    for dim in HOFSTEDE_DIMS:
        corr, pval = calculate_hofstede_correlation(
            adjusted_nps[signal_name], hofstede_df, dim
        )
        hofstede_corrs[dim] = {'correlation': corr, 'pvalue': pval}
    
    # Average absolute correlation
    avg_abs_corr = np.nanmean([abs(hofstede_corrs[d]['correlation']) for d in HOFSTEDE_DIMS])
    metrics['hofstede_correlation'] = avg_abs_corr
    metrics['hofstede_correlations'] = hofstede_corrs
    
    # 4. Agreement with stars (inter-model alignment)
    if signal_name != 'stars':
        agreement = calculate_agreement_score(
            df, 'stars', signal_info['score_col']
        )
        metrics['agreement_with_stars'] = agreement
        metrics['agreement'] = agreement  # For CRS-B
    else:
        # For stars, calculate agreement with other models
        agreements = []
        for other_signal, other_info in SENTIMENT_SIGNALS.items():
            if other_signal != 'stars' and other_info['score_col'] in df.columns:
                ag = calculate_agreement_score(
                    df, 'stars', other_info['score_col']
                )
                if not np.isnan(ag):
                    agreements.append(ag)
        metrics['agreement'] = np.nanmean(agreements) if agreements else 0.5
    
    # 5. Within-country consistency
    consistency = calculate_within_country_consistency(df, signal_info['score_col'])
    metrics['within_country_consistency'] = consistency
    
    # 6. Distribution alignment with stars
    if signal_name != 'stars':
        alignment = calculate_distribution_alignment(
            df['stars'], df[signal_info['score_col']]
        )
        metrics['distribution_alignment'] = alignment
    
    metrics_results[signal_name] = metrics
    
    # Print summary
    print(f"  Country Dispersion (Before): {initial_disp:.2f}")
    print(f"  Country Dispersion (After):  {adjusted_disp:.2f}")
    print(f"  Instability Reduction:        {reduction:.4f}")
    print(f"  Avg |Hofstede Correlation|:  {avg_abs_corr:.4f}")
    print(f"  Agreement Score:             {metrics.get('agreement', 'N/A'):.4f}")
    print(f"  Within-Country Consistency:   {consistency:.4f}")

print(f"\n{'='*80}")


In [ ]:
# Calculate CRS-B scores for each method
crs_scores = {}
crs_details = {}

print("=" * 80)
print("CALCULATING CRS-B SCORES")
print("=" * 80)

for signal_name, signal_info in SENTIMENT_SIGNALS.items():
    if signal_name not in metrics_results:
        continue
    
    metrics = metrics_results[signal_name]
    
    # Prepare metrics dict for CRS-B calculation
    crs_metrics = {
        'country_dispersion': metrics['country_dispersion'],
        'instability_reduction': metrics['instability_reduction'],
        'hofstede_correlation': metrics['hofstede_correlation'],
        'agreement': metrics.get('agreement', 0.5)
    }
    
    # Calculate CRS-B
    crs_score, crs_components = calculate_crs_b(crs_metrics, CONFIG['crs_initial_weights'])
    
    crs_scores[signal_info['name']] = crs_score
    crs_details[signal_name] = {
        'crs_score': crs_score,
        'components': crs_components,
        'metrics': crs_metrics
    }
    
    print(f"\n{signal_info['name']}:")
    print(f"  CRS-B Score: {crs_score:.4f}")
    print(f"  Components:")
    print(f"    - Country Score:      {crs_components['country_score']:.4f}")
    print(f"    - Instability Score:  {crs_components['instability_score']:.4f}")
    print(f"    - Hofstede Score:     {crs_components['hofstede_score']:.4f}")
    print(f"    - Agreement Score:    {crs_components['agreement_score']:.4f}")

# Find best method
if crs_scores:
    best_method = max(crs_scores, key=crs_scores.get)
    best_score = crs_scores[best_method]
    print(f"\n{'='*80}")
    print(f"BEST METHOD: {best_method} (CRS-B = {best_score:.4f})")
    print(f"{'='*80}")


## Validation: Prior Sensitivity Analysis


In [ ]:
# Prior sensitivity analysis (run for one signal as example)
print("=" * 80)
print("PRIOR SENSITIVITY ANALYSIS")
print("=" * 80)
print("Note: Running for stars signal as example (can be extended to all signals)")

prior_sensitivity_results = {}

if 'stars' in SENTIMENT_SIGNALS:
    signal_name = 'stars'
    signal_info = SENTIMENT_SIGNALS[signal_name]
    
    try:
        results = prior_sensitivity_analysis(df, signal_name, signal_info, CONFIG)
        prior_sensitivity_results[signal_name] = results
        
        print(f"\nPrior sensitivity analysis completed for {signal_info['name']}")
        print(f"Tested {len(results)} prior configurations")
    except Exception as e:
        print(f"Error in prior sensitivity analysis: {str(e)}")
        import traceback
        traceback.print_exc()


## Validation: Posterior Predictive Checks


In [ ]:
# Posterior predictive checks
print("=" * 80)
print("POSTERIOR PREDICTIVE CHECKS")
print("=" * 80)

ppc_results = {}

for signal_name, signal_info in SENTIMENT_SIGNALS.items():
    if signal_name not in fitted_models:
        continue
    
    try:
        mcmc = fitted_models[signal_name]
        ppc = posterior_predictive_check(mcmc, df, signal_info, country_to_idx)
        ppc_results[signal_name] = ppc
        
        print(f"\n{signal_info['name']}:")
        if ppc:
            print(f"  Observed Mean: {ppc.get('observed_mean', 'N/A'):.3f}")
            print(f"  Observed Std:  {ppc.get('observed_std', 'N/A'):.3f}")
            print(f"  Posterior Mean: {ppc.get('posterior_mean', 'N/A'):.3f}")
            print(f"  Posterior Std:  {ppc.get('posterior_std', 'N/A'):.3f}")
    except Exception as e:
        print(f"Error in PPC for {signal_info['name']}: {str(e)}")


## Validation: Leave-One-Country-Out (LOCO)


In [ ]:
# LOCO validation (run for one signal as example due to computational cost)
print("=" * 80)
print("LEAVE-ONE-COUNTRY-OUT (LOCO) VALIDATION")
print("=" * 80)
print("Note: Running for stars signal as example (can be extended to all signals)")

loco_results = {}

if 'stars' in SENTIMENT_SIGNALS:
    signal_name = 'stars'
    signal_info = SENTIMENT_SIGNALS[signal_name]
    
    try:
        results = leave_one_country_out_validation(
            df, signal_name, signal_info, prior_config, CONFIG
        )
        loco_results[signal_name] = results
        
        print(f"\nLOCO validation completed for {signal_info['name']}")
        print(f"Validated on {len(results)} countries")
        for country, metrics in results.items():
            print(f"  {country}: test_size={metrics.get('test_size', 'N/A')}, "
                  f"test_mean={metrics.get('test_mean', 'N/A'):.3f}")
    except Exception as e:
        print(f"Error in LOCO validation: {str(e)}")
        import traceback
        traceback.print_exc()


## Visualizations


In [ ]:
# Generate visualizations
print("=" * 80)
print("GENERATING VISUALIZATIONS")
print("=" * 80)

# 1. Sentiment distributions
try:
    plot_sentiment_distributions(df, SENTIMENT_SIGNALS, OUTPUT_DIR, TIMESTAMP)
    print("✓ Sentiment distributions plot saved")
except Exception as e:
    print(f"✗ Error plotting sentiment distributions: {str(e)}")

# 2. NPS comparison
try:
    plot_nps_comparison(initial_nps, adjusted_nps, SENTIMENT_SIGNALS, OUTPUT_DIR, TIMESTAMP)
    print("✓ NPS comparison plot saved")
except Exception as e:
    print(f"✗ Error plotting NPS comparison: {str(e)}")

# 3. CRS-B comparison
try:
    plot_crs_comparison(crs_scores, OUTPUT_DIR, TIMESTAMP)
    print("✓ CRS-B comparison plot saved")
except Exception as e:
    print(f"✗ Error plotting CRS-B comparison: {str(e)}")

print(f"\nAll plots saved to: {OUTPUT_DIR}")


## Create Comprehensive Metrics Table


In [ ]:
# Create comprehensive metrics table
metrics_table_data = []

for signal_name, signal_info in SENTIMENT_SIGNALS.items():
    if signal_name not in metrics_results:
        continue
    
    metrics = metrics_results[signal_name]
    crs_info = crs_details.get(signal_name, {})
    
    row = {
        'Method': signal_info['name'],
        'Country Dispersion (Before)': f"{metrics.get('country_dispersion_before', 0):.2f}",
        'Country Dispersion (After)': f"{metrics.get('country_dispersion_after', 0):.2f}",
        'Instability Reduction': f"{metrics.get('instability_reduction', 0):.4f}",
        'Avg |Hofstede Correlation|': f"{metrics.get('hofstede_correlation', 0):.4f}",
        'Agreement Score': f"{metrics.get('agreement', 0):.4f}",
        'Within-Country Consistency': f"{metrics.get('within_country_consistency', 0):.4f}",
        'CRS-B Score': f"{crs_info.get('crs_score', 0):.4f}"
    }
    
    metrics_table_data.append(row)

metrics_df = pd.DataFrame(metrics_table_data)
print("\nComprehensive Metrics Table:")
print("=" * 120)
print(metrics_df.to_string(index=False))
print("=" * 120)

# Save metrics table
metrics_df.to_csv(OUTPUT_DIR / f"metrics_table_{TIMESTAMP}.csv", index=False)
print(f"\nMetrics table saved to: {OUTPUT_DIR / f'metrics_table_{TIMESTAMP}.csv'}")


## Generate Comprehensive Report


In [ ]:
# Prepare results dictionary for report
elapsed_time = time.time() - START_TIME

results_dict = {
    'models': fitted_models,
    'initial_nps': initial_nps,
    'adjusted_nps': adjusted_nps,
    'metrics_results': metrics_results,
    'crs_scores': crs_scores,
    'crs_details': crs_details,
    'prior_sensitivity': prior_sensitivity_results,
    'ppc_results': ppc_results,
    'loco_results': loco_results,
    'metrics_table': metrics_df.to_string(index=False).split('\n'),
    'observations': [
        f"Total of {len(fitted_models)} Bayesian models were successfully fitted.",
        f"Best performing method by CRS-B: {max(crs_scores, key=crs_scores.get) if crs_scores else 'N/A'}",
        f"Average instability reduction across all methods: {np.mean([m.get('instability_reduction', 0) for m in metrics_results.values()]):.4f}",
        f"Cultural bias adjustment shows improvement in NPS reliability across countries.",
        f"Country-wise NPS dispersion reduced after adjustment for most methods."
    ]
}

# Generate report
report_path = generate_comprehensive_report(
    results_dict, CONFIG, TIMESTAMP, OUTPUT_DIR, elapsed_time
)

print(f"\n{'='*80}")
print("ANALYSIS COMPLETE")
print(f"{'='*80}")
print(f"Total execution time: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
print(f"Results saved to: {OUTPUT_DIR}")
print(f"Report: {report_path}")
print(f"{'='*80}")
